#### Embedding Our Dataset

In [ ]:
# Download the ingest.py file in this 02-vector_search directory
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

--2026-06-29 13:04:02--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 888 [text/plain]
Saving to: ‘ingest.py’

ingest.py           100%[===================>]     888  --.-KB/s    in 0s      

2026-06-29 13:04:02 (32.6 MB/s) - ‘ingest.py’ saved [888/888]



In [ ]:
# Load the data
from ingest import load_faq_data
documents = load_faq_data()

In [8]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
# convert the array of object into an array of string
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [7]:
# Generating embeddings
from tqdm import tqdm

In [ ]:
# Chunk the dataset into batches of 50 and encode each batch
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
  batch = texts[i:i + batch_size]
  batch_vectors = model.encode(batch)
  vectors.extend(batch_vectors)

len(vectors)  

100%|██████████| 27/27 [01:58<00:00,  4.37s/it]


1350

In [11]:
# We turn them into a 2-dimensional array (matrix) where
import numpy as np
X = np.array(vectors)

#### Vector Search

In [12]:
# Scoring documents
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [13]:
# Next, we compute the dot product against all documents:
scores = X.dot(v_query)

In [ ]:
# Best match. The highest score is the most similar document:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.762941))

In [16]:
# Let's see which document it 
documents[idx]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

In [21]:
# Top 5 results. np.argsort sorts from lowest to highest, so the last 5 are the top ones:
top5 = np.argsort(scores)[-5:]
top5

array([  7, 538, 907, 625,   2])

In [22]:
# They come out smallest-first, so we reverse them to get the highest first:
top5 = top5[::-1]
top5

array([  2, 625, 907, 538,   7])

In [23]:
scores[top5]

array([0.762941  , 0.7579371 , 0.7192132 , 0.6536312 , 0.56009996],
      dtype=float32)

In [24]:
top5 = np.argsort(-scores)[:5]

In [25]:
# Let's read off the actual documents
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.7579371
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.7192132
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 